# 15. CUDA Execution Model | CUDA 执行模型

**难度：** Medium | **环境：** GPU optional | **标签：** `算子编程`, `CUDA`, `执行模型` | **目标人群：** 编译与系统入门者

![CUDA 执行层级与资源映射](../docs/public/01_Hardware_Math_and_Systems/15_cuda_execution_hierarchy.svg)

---

## 本节导读

CUDA kernel 会把同一项张量任务分配到 Grid、Block、Warp 和 Thread 四个层级。层级变化会影响覆盖范围、尾部空闲线程、协作边界和可用片上资源。

本节从任务覆盖、Warp 打包和 Block 资源约束建立执行账本。完成后，你应能从元素数量推导 launch 几何，并判断一个 Block 配置是否满足线程、共享内存和寄存器限制；这些能力直接支撑算子优化路线及 Part 03–04 的 Kernel 实现。

**关键词：** `grid`, `block`, `warp`

---

## 前置阅读

**导语：** 先把 GPU 上的执行层级对齐，再看 kernel 为什么要按 block 组织会更顺。

- [Group 1B: Single-Device Execution, Memory and Data Access | 1B: 单设备执行、内存与访存](./1B.md)
- [Group 1D: Execution Models, Asynchrony and Operator Mapping | 1D: 执行模型、异步调度与算子映射](./1D.md)
- [13. Profiling and Bottleneck Analysis | 性能分析与瓶颈定位](./13_Profiling_and_Bottleneck_Analysis.ipynb)


## Q1：GPU 上的 kernel 为什么不是“一个线程在跑”？


GPU 的执行不是单线程串行模型，而是分层并行模型。

- **Grid**：一次 kernel launch 对应的整体任务集合。
- **Block**：同一批线程的协作单元，通常共享片上资源。
- **Warp**：硬件真正调度和执行的基本粒度，通常是 32 个线程一组。
- **Thread**：最小执行单元。

理解这个层次很重要，因为 kernel 的性能不是只看“写了多少代码”，而是看任务被切成什么粒度、线程之间怎么协作、以及硬件怎么把这些线程排进执行队列。

```mermaid
flowchart TD
    A[Kernel Launch] --> B[Grid]
    B --> C[Block]
    C --> D[Warp]
    D --> E[Thread]
    C --- F[Shared resources]
    D --- G[SIMT execution]
```

所以 GPU 不是“很多线程在随便跑”，而是一个有明确层级和调度约束的执行系统。


### Q1 机制计算器：从元素数量推导 Grid、Block 与 Warp

先记住 grid / block / warp / thread 的关系，再去看 kernel 代码会更顺。

In [ ]:
def launch_geometry(num_elements, threads_per_block, warp_size=32):
    """计算一维 kernel 的 block、warp 和尾块有效线程。"""
    if num_elements <= 0 or threads_per_block <= 0 or warp_size <= 0:
        raise ValueError('sizes must be positive')
    blocks = (num_elements + threads_per_block - 1) // threads_per_block
    warps_per_block = (threads_per_block + warp_size - 1) // warp_size
    launched_threads = blocks * threads_per_block
    return {
        'grid_blocks': blocks,
        'warps_per_block': warps_per_block,
        'total_warps': blocks * warps_per_block,
        'launched_threads': launched_threads,
        'inactive_tail_threads': launched_threads - num_elements,
    }

for threads_per_block in [64, 128, 256]:
    print(threads_per_block, '->', launch_geometry(1000, threads_per_block))
assert launch_geometry(1000, 256)['inactive_tail_threads'] == 24


## Q2：block 和 warp 的分工是什么，为什么 shared memory 这么重要？


Block 是线程协作的主要边界，Warp 是真正执行时最常见的同步单位。

- **Block** 适合做局部协作、数据复用和共享内存管理。
- **Warp** 适合做更细粒度的同步和寄存器级协作。

Shared memory 之所以重要，是因为它位于 HBM 和寄存器之间，速度更快，且适合 block 内线程共享数据。很多 kernel 优化的关键，不是把计算写得更多，而是把数据尽量留在 shared memory 里，减少反复访问 HBM。

```mermaid
flowchart LR
    HBM[HBM / Global Memory] --> SMEM[Shared Memory]
    SMEM --> REG[Registers]
    Block[Block 内线程] --> SMEM
    Warp[Warp 同步] --> REG
    Block -. reuse .-> HBM
```

这就是为什么后面看 Triton block 模型或 FlashAttention 时，shared memory 总会反复出现：它是 block 协作和数据复用的核心舞台。


### Q2 流量账本：Block 复用减少了多少 HBM 读取

先分清 block、warp 和 shared memory 各自负责什么。

In [ ]:
def block_reuse_traffic(blocks, tile_elements, bytes_per_element, reuse_count):
    """比较每次从 HBM 重读与每个 Block 只加载一次的字节量。"""
    if min(blocks, tile_elements, bytes_per_element, reuse_count) <= 0:
        raise ValueError('all inputs must be positive')
    tile_bytes = tile_elements * bytes_per_element
    baseline_hbm_bytes = blocks * tile_bytes * reuse_count
    tiled_hbm_bytes = blocks * tile_bytes
    return {
        'baseline_hbm_bytes': baseline_hbm_bytes,
        'tiled_hbm_bytes': tiled_hbm_bytes,
        'hbm_bytes_saved': baseline_hbm_bytes - tiled_hbm_bytes,
        'shared_bytes_per_block': tile_bytes,
    }

for reuse_count in [1, 2, 4]:
    print('reuse', reuse_count, '->', block_reuse_traffic(8, 4096, 2, reuse_count))
assert block_reuse_traffic(8, 4096, 2, 1)['hbm_bytes_saved'] == 0


## Q3：为什么先理解执行模型，后面学 Triton 和 CUDA 会快很多？


Triton 和 CUDA 不是先学语法再学性能，而是先理解执行模型，再把代码映射到硬件层级。

如果不知道 grid / block / warp 的关系，就很难理解：
- 为什么某些 kernel 需要按 block 组织数据；
- 为什么 shared memory 能带来收益；
- 为什么某些优化要避免过多同步；
- 为什么一个小改动会明显改变吞吐。

```mermaid
flowchart LR
    A[执行模型] --> B[数据如何切块]
    A --> C[线程如何协作]
    A --> D[资源如何分配]
    B --> E[Triton block / tile]
    C --> F[Cuda kernel sync]
    D --> G[shared memory / registers]
```

执行模型是后面所有 kernel 优化的共同前提。先把这层讲清楚，后面的 Triton block、mask、tile、fusion 才不会像单纯的术语堆叠。


### Q3 资源门槛：一个 Block 配置能否进入候选

看到 kernel 代码时，先问自己它对应的是哪一层。

In [ ]:
def block_resource_gate(threads_per_block, shared_bytes_per_block, registers_per_thread, limits):
    """检查线程、共享内存和寄存器三个 Block 资源硬约束。"""
    if min(threads_per_block, shared_bytes_per_block, registers_per_thread) < 0:
        raise ValueError('resource usage must be non-negative')
    reasons = []
    if threads_per_block > limits['max_threads_per_block']:
        reasons.append('threads_per_block')
    if shared_bytes_per_block > limits['max_shared_bytes_per_block']:
        reasons.append('shared_memory')
    register_bytes = threads_per_block * registers_per_thread * 4
    if register_bytes > limits['register_file_bytes_per_block']:
        reasons.append('register_file')
    return {'eligible': not reasons, 'reasons': reasons, 'register_bytes_per_block': register_bytes}

limits = {'max_threads_per_block': 1024, 'max_shared_bytes_per_block': 64 * 1024, 'register_file_bytes_per_block': 256 * 1024}
valid = block_resource_gate(256, 32 * 1024, 64, limits)
invalid = block_resource_gate(2048, 96 * 1024, 64, limits)
assert valid['eligible'] is True
assert invalid['eligible'] is False
print(valid)
print(invalid)


## ⚠️ 常见误区

- GPU 不是“一个线程一个线程顺着跑”，而是分层并行调度。
- block 和 warp 不是同一层概念，前者偏组织，后者偏执行。
- shared memory 不是越多越好，而是要配合 block 内的数据复用。
- 先懂执行模型，再看 kernel 代码，理解会快很多。

## 相关阅读

**导语：** 把执行模型和 Triton 的 block 思维放一起看，后面写 kernel 会更稳。

- [Part 03: Triton Kernel Development | 第三部分：Triton 算子开发](../03_Triton_Kernels/intro.md)
- [01. Triton 入门与 Hello World：向量加法 (Vector Addition)](../03_Triton_Kernels/01_Triton_Vector_Addition.ipynb)
- [04. Triton 矩阵乘法 (GEMM) 与自动调优 (Autotune)](../03_Triton_Kernels/04_Triton_GEMM_Tutorial.ipynb)
---